#  Brain Tumor Detection — Grad-CAM Explainability

Deep learning models are often called "black boxes" — they make predictions without explaining why.
This is especially problematic in medical AI where trust and interpretability are critical. 

**Grad-CAM (Gradient-weighted Class Activation Mapping)** Transforms deep learning models into interprretable. They highlight the tumor region of the MRI scan by *showing a heatmap of this region
the model focused on* when making its prediction.

A trustworthy prediction should highlight the actual tumor region — not random background areas.

## 1. Imports & Setup

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from pathlib import Path

sys.path.append('../src')
from gradcam import GradCAM

CLASSES     = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = len(CLASSES)
IMG_SIZE    = 224
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR  = Path('../models')
RESULTS_DIR = Path('../results')
DATA_DIR    = Path('../data')
RESULTS_DIR.mkdir(exist_ok=True)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f'Device: {DEVICE}')

Device: cuda


## 2. Load Model

In [ ]:
model = models.resnet50(weights=None)
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(256, NUM_CLASSES)
)
model.load_state_dict(torch.load(MODELS_DIR / 'best_model.pth', map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print(' Model loaded!')

✅ Model loaded!


## 3. How Grad-CAM Works

```
Input MRI
    ↓
Forward pass through ResNet50
    ↓
Prediction (e.g. Glioma)
    ↓
Backward pass — compute gradients w.r.t. layer4 activations
    ↓
Global average pool the gradients → importance weights per feature map
    ↓
Weighted sum of feature maps → raw heatmap
    ↓
Apply ReLU (keep only positive contributions)
    ↓
Resize to input image size → overlay
```

The hook is placed at `model.layer4[-1]` — the last residual block of ResNet50,
which captures the highest-level semantic features.

## 4. Grad-CAM Across All 4 Classes

We pick one sample from each class and visualize the heatmap side by side.

In [ ]:
def get_gradcam_overlay(image_path, model):
    img        = Image.open(image_path).convert('RGB')
    img_tensor = transform(img)
    gradcam    = GradCAM(model)
    heatmap, class_idx = gradcam.generate(img_tensor)

    heatmap_img = Image.fromarray(np.uint8(255 * heatmap))
    heatmap_img = heatmap_img.resize(img.size, Image.BILINEAR)
    heatmap_np  = np.array(heatmap_img) / 255.0

    colormap = plt.cm.jet(heatmap_np)[:, :, :3]
    img_np   = np.array(img) / 255.0
    overlay  = np.clip(0.6 * img_np + 0.4 * colormap, 0, 1)

    return img, heatmap_np, overlay, CLASSES[class_idx]


fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle('Grad-CAM — One Sample per Class', fontsize=16, fontweight='bold')

col_titles = ['Original MRI', 'Heatmap', 'Overlay']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight='bold')

for row, cls in enumerate(CLASSES):
    cls_dir    = DATA_DIR / 'testing' / cls
    img_file   = Path(cls_dir) / os.listdir(cls_dir)[0]
    img, heatmap, overlay, predicted = get_gradcam_overlay(img_file, model)

    correct = predicted == cls
    label   = f'Actual: {cls.capitalize()}\nPredicted: {predicted.capitalize()}'
    color   = 'green' if correct else 'red'

    axes[row][0].imshow(img)
    axes[row][0].set_ylabel(label, fontsize=9, color=color, rotation=0,
                            labelpad=100, va='center')
    axes[row][0].axis('off')

    axes[row][1].imshow(heatmap, cmap='jet')
    axes[row][1].axis('off')

    axes[row][2].imshow(overlay)
    axes[row][2].axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'gradcam_all_classes.png', dpi=150, bbox_inches='tight')
plt.close()
print(' Saved → results/gradcam_all_classes.png')

✅ Saved → results/gradcam_all_classes.png


## 5. Multiple Samples per Class

One sample isn't enough — let's look at 3 samples per class to see if the attention is consistent.

In [ ]:
SAMPLES_PER_CLASS = 3

fig, axes = plt.subplots(NUM_CLASSES * SAMPLES_PER_CLASS, 3,
                          figsize=(12, NUM_CLASSES * SAMPLES_PER_CLASS * 4))
fig.suptitle('Grad-CAM — 3 Samples per Class', fontsize=16, fontweight='bold')

for row_cls, cls in enumerate(CLASSES):
    cls_dir  = DATA_DIR / 'testing' / cls
    img_files = os.listdir(cls_dir)[:SAMPLES_PER_CLASS]

    for sample_i, img_name in enumerate(img_files):
        row = row_cls * SAMPLES_PER_CLASS + sample_i
        img_path = Path(cls_dir) / img_name
        img, heatmap, overlay, predicted = get_gradcam_overlay(img_path, model)

        correct = predicted == cls
        color   = 'green' if correct else 'red'

        axes[row][0].imshow(img)
        axes[row][0].set_ylabel(f'{cls.capitalize()} #{sample_i+1}\n→ {predicted.capitalize()}',
                                fontsize=8, color=color, rotation=0,
                                labelpad=90, va='center')
        axes[row][0].axis('off')
        axes[row][1].imshow(heatmap, cmap='jet')
        axes[row][1].axis('off')
        axes[row][2].imshow(overlay)
        axes[row][2].axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'gradcam_multi_sample.png', dpi=150, bbox_inches='tight')
plt.close()
print(' Saved → results/gradcam_multi_sample.png')

✅ Saved → results/gradcam_multi_sample.png


## 6. Observations

- **Glioma:** The heatmap should highlight irregular, diffuse regions in the brain tissue.
- **Meningioma:** Attention should focus near the outer edges/meninges of the brain.
- **No Tumor:** Heatmap should be more scattered with no single strong region.
- **Pituitary:** Attention should concentrate at the base/center of the brain.

If the heatmaps match these expectations, the model is learning medically meaningful features — not just texture or background artifacts.